# Transaction Data Cleaning

**Vintage Sampling Media Resale | Jan 2025 – Jun 2026**

Prepares raw CSV export for CLV and RFM analysis.

In [ ]:
import pandas as pd
import numpy as np
import hashlib # for anonymizing buyer usernames

def anonymize_username(username):
    return hashlib.md5(str(username).encode()).hexdigest()[:10]

1. Load Data

In [ ]:
df = pd.read_csv("../data/transactions.csv")

df['buyer_id'] = df['Buyer Username'].apply(anonymize_username) # Anonymize buyer usernames
df = df.drop(columns=['Buyer Username']) # Drop buyer usernames

In [ ]:
# Inspect raw data
print("Raw shape:", df.shape) 
print(df.head(10))

2. Rename columns

In [ ]:
df.columns = df.columns.str.strip()  # strip whitespace from headers

df = df.rename(columns={
    "Order Number":          "order_number",
    "Buyer State":           "buyer_state",
    "Buyer Country":         "buyer_country",
    "Item Title":            "item_title",
    "Sold For":              "sold_for",
    "Shipping And Handling": "shipping",
    "Total Price":           "total_price",
    "Sale Date":             "sale_date",
    "Paid On Date":          "paid_on_date",
})

df = df.drop(columns=['shipping', 'total_price']) # Drop shipping column as it will skew the analysis as shipping internationally is much more expensive than shipping to the US, and we want to focus on the actual sale price of the items. Since total_price is the sum of sold_for and shipping, we can drop it as well.

3. Forward fill buyer info into blank multi-item rows

In [ ]:
fill_cols = ["buyer_id", "buyer_state", "buyer_country", "order_number",
             "paid_on_date", "sale_date"]

df[fill_cols] = df[fill_cols].replace("", np.nan)  # blank strings to NaN
df[fill_cols] = df.groupby("order_number")[fill_cols].ffill() # forward fill by order_number to avoid filling in buyer info from other transactions

df = df[df["item_title"].notna() & (df["item_title"].str.strip() != "")] # drop transactions with no item title, they are aggrtegated transactions with no item-level data

4. Clean currency columns

In [ ]:
for col in ["sold_for", "shipping", "total_price"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("[$,]", "", regex=True)
            .str.strip()
            .replace("", np.nan)
            .astype(float)
        )

 5. Parse dates

In [ ]:
df["sale_date"] = pd.to_datetime(df["sale_date"], format="%b-%d-%y", errors="coerce")
df["paid_on_date"] = pd.to_datetime(df["paid_on_date"], format="%b-%d-%y", errors="coerce")
 
# Use paid_on_date as the canonical transaction date
df["order_date"] = df["paid_on_date"].fillna(df["sale_date"])

6. Aggregate to order level

In [ ]:
order_agg = df.groupby("order_number").agg(
    buyer_id=("buyer_id", "first"),
    buyer_state=("buyer_state",    "first"),
    buyer_country=("buyer_country", "first"),
    order_date=("order_date",      "first"),
    order_value=("sold_for",       "sum"),    # sum of item prices
    n_items=("item_title",         "count"),
).reset_index()

7. Drop rows with missing buyer or dat

In [ ]:
order_agg = order_agg.dropna(subset=["buyer_id", "order_date", "order_value"])
order_agg["order_value"] = order_agg["order_value"].round(2) # fix floating point precision issues

8. Sanity checks 

In [ ]:
# Inspect processed data
print("\nCleaned order-level shape:", order_agg.shape)

print("\nDate range:",
      order_agg["order_date"].min(), "→", order_agg["order_date"].max())

print("\nUnique buyers:", order_agg["buyer_id"].nunique())

print("\nTop countries:\n", order_agg["buyer_country"].value_counts().head(10))

print("\nTop states (US):\n",
      order_agg[order_agg["buyer_country"] == "United States"]["buyer_state"]
      .value_counts())

print("\nUnique states (US):\n",
      order_agg[order_agg["buyer_country"] == "United States"]["buyer_state"]
      .nunique())

print("\nOrder value summary:\n", order_agg["order_value"].describe())

9. Save cleaned data

In [ ]:
order_agg.to_csv("../data/orders_clean.csv", index=False)
print("\nSaved: orders_clean.csv")